# SCARED — Exploración de archivos y formatos

**¿Qué hace este notebook?**  
Recorre los ZIPs del dataset SCARED sin descomprimirlos al disco y responde tres preguntas básicas:

1. ¿Qué hay en cada ZIP? ¿Cuántos keyframes, qué archivos tiene cada uno?
2. ¿Cómo se ve cada tipo de archivo? (PNG, TIFF, OBJ, YAML, MP4, TAR.GZ)
3. ¿Qué dimensiones y rangos de valores tienen los datos más importantes?

**Lo que NO hace este notebook:** análisis profundo, métricas, comparaciones entre keyframes.  
Eso viene en notebooks posteriores.

---

### Antes de correr el notebook

Solo necesitas cambiar `SCARED_DIR` en la celda de configuración si tus ZIPs están en otro lugar.  
El ambiente de conda que usamos se llama `computer_vision`.

---
## Estructura de la carpeta `E:/scared_raw`

> **Fuente:** [README.md oficial del dataset](e:/scared_raw/README.md) — *MICCAI Challenge 2019 Dataset*

El README describe el dataset original del challenge como "3 datasets from different pigs, each containing 5 keyframes". La versión completa descargada aquí tiene **9 datasets** (la versión extendida publicada en HuggingFace), pero la estructura interna es idéntica.

```
E:/scared_raw/
│
├── dataset_1.zip   (12.8 GB)
├── dataset_2.zip   (22.9 GB)  ← un ZIP por dataset (cerdo distinto o sesión distinta)
├── ...
├── dataset_9.zip   (16.5 GB)
│
├── test_dataset_8.zip   (240 MB)  ← versión reducida para pruebas rápidas
├── test_dataset_9.zip   (378 MB)
│
├── README.md              ← descripción oficial del challenge MICCAI 2019
├── download_log.txt
└── ...scripts de descarga...
```

Cada ZIP contiene **5 keyframes**. Según el README: *"Keyframes are all single captures with a unique positioning of the endoscope and several positions of the structured light illuminator."*

```
dataset_1.zip
└── dataset_1/
    ├── keyframe_1/
    │   ├── Left_Image.png              (2.5 MB)  ← "left camera view used when the structured
    │   │                                             light patterns were captured" [README]
    │   ├── Right_Image.png             (2.5 MB)  ← cámara derecha (par estéreo)
    │   ├── left_depth_map.tiff        (15.7 MB)  ← "point cloud as seen by the left camera.
    │   │                                             Each pixel contains an (X,Y,Z) coordinate
    │   │                                             in left camera space" [README]
    │   ├── right_depth_map.tiff       (15.7 MB)  ← mismo formato, cámara derecha
    │   ├── point_cloud.obj            (63.6 MB)  ← "3D vertex model, viewable in Blender/Meshlab" [README]
    │   ├── endoscope_calibration.yaml  (~0 MB)   ← "approximate calibration, may be inaccurate.
    │   │                                             Methods must be robust to calibration errors" [README]
    │   └── data/
    │       ├── rgb.mp4                (33.0 MB)  ← "frames stacked left above right, captured
    │       │                                         after the structured light sequence" [README]
    │       │                                         ⚠️ No abre en VLC/QuickTime, usar OpenCV o ffmpeg
    │       ├── frame_data.tar.gz       (~0 MB)   ← JSONs con nombre frame_data%06d.json:
    │       │                                         "camera transform relative to the keyframe
    │       │                                         position + calibration data" [README]
    │       └── scene_points.tar.gz  (2 611 MB)  ← TIFFs con nombre scene_points%06d.tiff:
    │                                               "vertex coordinates warped from the keyframe.
    │                                               Pixels without GT depth contain (0,0,0)" [README]
    │                                               ⚠️ El archivo más pesado del dataset
    ├── keyframe_2/   (misma estructura)
    ├── keyframe_3/   (misma estructura)
    ├── keyframe_4/   ⚠️ sin carpeta data/ — "interpolation is missing due to a logging error" [README]
    └── keyframe_5/   (misma estructura, sin data/ por ser el último keyframe)
```

**Notas técnicas del README:**

- Los depth maps warpeados en `scene_points.tar.gz` **no son GT perfecto** — *"due to errors in the kinematics or synchronization issues, there will be small errors in the warped vertex positions"*. Úsalos con precaución.
- La calibración (`endoscope_calibration.yaml`) es **aproximada**. Los modelos deben ser robustos a errores en ella.
- Los archivos `.tiff` almacenan valores flotantes (float32) — requieren `tifffile` (`pip install tifffile`), no abren bien con visores de imagen normales.
- El video `rgb.mp4` tiene los frames de ambas cámaras **apilados verticalmente** (izquierda arriba, derecha abajo).

> **¿Qué usamos para el proyecto de corrección de luz?**  
> Principalmente `Left_Image.png` (entrada del modelo) y `left_depth_map.tiff` (supervisión GT).  
> El resto sirve para análisis avanzados y entrenamiento con datos de video.

In [ ]:
from IPython.display import SVG, display

svg1 = """
<svg width="760" height="620" xmlns="http://www.w3.org/2000/svg"
     font-family="'Segoe UI', Arial, sans-serif">

  <!-- Fondo blanco -->
  <rect width="760" height="620" rx="12" fill="#ffffff" stroke="#e2e8f0" stroke-width="1.5"/>

  <!-- Banda de título -->
  <rect x="0" y="0" width="760" height="50" rx="12" fill="#1e3a5f"/>
  <rect x="0" y="38" width="760" height="12" fill="#1e3a5f"/>
  <text x="380" y="30" text-anchor="middle" fill="#ffffff" font-size="14" font-weight="bold">
    E:/scared_raw/ — vista general del dataset
  </text>
  <text x="380" y="60" text-anchor="middle" fill="#64748b" font-size="10.5">
    MICCAI Endoscopic Vision Challenge 2019 · versión extendida (HuggingFace)
  </text>

  <!-- ════════════════════════════════
       COLUMNA IZQUIERDA  410 px
       x=18  ancho=410  →  hasta x=428
       ════════════════════════════════ -->

  <!-- Nodo raíz -->
  <rect x="18" y="78" width="220" height="34" rx="6"
        fill="none" stroke="#1e3a5f" stroke-width="1.5"/>
  <text x="30" y="101" fill="#1e3a5f" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">📂 E:/scared_raw/</text>

  <line x1="46" y1="112" x2="46" y2="406" stroke="#cbd5e1" stroke-width="1.5"/>

  <!-- dataset_1 -->
  <line x1="46" y1="140" x2="72" y2="140" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="122" width="356" height="36" rx="5" fill="none" stroke="#bbf7d0" stroke-width="1"/>
  <rect x="72" y="122" width="5" height="36" rx="2" fill="#16a34a"/>
  <text x="88" y="137" fill="#15803d" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">dataset_1.zip</text>
  <text x="88" y="151" fill="#64748b" font-size="10">keyframe_1…keyframe_5 · 12.8 GB</text>

  <!-- dataset_2 -->
  <line x1="46" y1="174" x2="72" y2="174" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="166" width="356" height="36" rx="5" fill="none" stroke="#bbf7d0" stroke-width="1"/>
  <rect x="72" y="166" width="5" height="36" rx="2" fill="#16a34a"/>
  <text x="88" y="181" fill="#15803d" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">dataset_2.zip</text>
  <text x="88" y="195" fill="#64748b" font-size="10">keyframe_1…keyframe_5 · 22.9 GB</text>

  <!-- puntos suspensivos -->
  <line x1="46" y1="216" x2="72" y2="216" stroke="#cbd5e1" stroke-width="1.5"/>
  <text x="88" y="222" fill="#94a3b8" font-size="12">⋮  dataset_3.zip … dataset_7.zip</text>

  <!-- dataset_8 -->
  <line x1="46" y1="244" x2="72" y2="244" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="232" width="356" height="36" rx="5" fill="none" stroke="#bae6fd" stroke-width="1"/>
  <rect x="72" y="232" width="5" height="36" rx="2" fill="#0891b2"/>
  <text x="88" y="247" fill="#0e7490" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">dataset_8.zip</text>
  <text x="88" y="261" fill="#64748b" font-size="10">keyframe_0…keyframe_4 · 32.4 GB</text>

  <!-- dataset_9 -->
  <line x1="46" y1="284" x2="72" y2="284" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="276" width="356" height="36" rx="5" fill="none" stroke="#bae6fd" stroke-width="1"/>
  <rect x="72" y="276" width="5" height="36" rx="2" fill="#0891b2"/>
  <text x="88" y="291" fill="#0e7490" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">dataset_9.zip</text>
  <text x="88" y="305" fill="#64748b" font-size="10">keyframe_0…keyframe_4 · 16.5 GB</text>

  <line x1="72" y1="322" x2="434" y2="322" stroke="#f1f5f9" stroke-width="1"/>

  <!-- test_dataset_8 -->
  <line x1="46" y1="342" x2="72" y2="342" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="328" width="356" height="36" rx="5" fill="none" stroke="#e9d5ff" stroke-width="1" stroke-dasharray="5,3"/>
  <rect x="72" y="328" width="5" height="36" rx="2" fill="#7c3aed"/>
  <text x="88" y="343" fill="#6d28d9" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">test_dataset_8.zip</text>
  <text x="88" y="357" fill="#64748b" font-size="10">Sin scene_points.tar.gz · 240 MB</text>

  <!-- README.md -->
  <line x1="46" y1="380" x2="72" y2="380" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="72" y="368" width="356" height="36" rx="5" fill="none" stroke="#cbd5e1" stroke-width="1"/>
  <rect x="72" y="368" width="5" height="36" rx="2" fill="#64748b"/>
  <text x="88" y="383" fill="#475569" font-size="11.5" font-weight="bold"
        font-family="'Courier New', monospace">README.md · *.ps1 · code.zip</text>
  <text x="88" y="397" fill="#64748b" font-size="10">Descripción oficial del challenge · scripts de descarga</text>

  <!-- Callout advertencia -->
  <rect x="18" y="416" width="410" height="66" rx="6"
        fill="#fef9c3" stroke="#fde047" stroke-width="1"/>
  <text x="32" y="434" fill="#854d0e" font-size="11" font-weight="bold">⚠  Nombres de keyframe inconsistentes entre datasets</text>
  <text x="32" y="450" fill="#713f12" font-size="10">dataset_1–7 → keyframe_1…keyframe_5  (índice desde 1)</text>
  <text x="32" y="464" fill="#713f12" font-size="10">dataset_8–9 → keyframe_0…keyframe_4  (índice desde 0)</text>
  <text x="32" y="478" fill="#92400e" font-size="9.5" font-style="italic">Usar siempre catalog[ds]['keyframes'] — no hardcodear nombres</text>

  <!-- ════════════════════════════════
       COLUMNA DERECHA  290 px
       x=442  ancho=300  →  hasta x=742
       ════════════════════════════════ -->
  <rect x="442" y="78" width="300" height="404" rx="8"
        fill="#f8fafc" stroke="#e2e8f0" stroke-width="1.2"/>

  <text x="592" y="104" text-anchor="middle" fill="#1e3a5f" font-size="12" font-weight="bold">
    Leyenda
  </text>
  <line x1="460" y1="112" x2="724" y2="112" stroke="#e2e8f0" stroke-width="1"/>

  <!-- Verde -->
  <rect x="460" y="122" width="5" height="18" rx="1" fill="#16a34a"/>
  <text x="474" y="136" fill="#15803d" font-size="10.5" font-weight="bold">dataset_1–7</text>
  <text x="474" y="150" fill="#64748b" font-size="9.5">keyframe_1…5 · incluye data/</text>
  <text x="474" y="163" fill="#64748b" font-size="9.5">RGB + depth + OBJ + video + poses</text>
  <line x1="460" y1="173" x2="724" y2="173" stroke="#f1f5f9" stroke-width="1"/>

  <!-- Cyan -->
  <rect x="460" y="182" width="5" height="18" rx="1" fill="#0891b2"/>
  <text x="474" y="196" fill="#0e7490" font-size="10.5" font-weight="bold">dataset_8–9</text>
  <text x="474" y="210" fill="#64748b" font-size="9.5">keyframe_0…4 · misma estructura</text>
  <text x="474" y="223" fill="#e11d48" font-size="9.5" font-weight="bold">⚠ Índice desde 0, no desde 1</text>
  <line x1="460" y1="233" x2="724" y2="233" stroke="#f1f5f9" stroke-width="1"/>

  <!-- Morado -->
  <rect x="460" y="242" width="5" height="18" rx="1" fill="#7c3aed"/>
  <text x="474" y="256" fill="#6d28d9" font-size="10.5" font-weight="bold">test_dataset_N</text>
  <text x="474" y="270" fill="#64748b" font-size="9.5">Sin scene_points.tar.gz</text>
  <text x="474" y="283" fill="#64748b" font-size="9.5">Para desarrollo y pruebas rápidas</text>
  <line x1="460" y1="293" x2="724" y2="293" stroke="#f1f5f9" stroke-width="1"/>

  <!-- Gris -->
  <rect x="460" y="302" width="5" height="18" rx="1" fill="#64748b"/>
  <text x="474" y="316" fill="#475569" font-size="10.5" font-weight="bold">Archivos auxiliares</text>
  <text x="474" y="330" fill="#64748b" font-size="9.5">README.md · scripts PowerShell</text>
  <line x1="460" y1="340" x2="724" y2="340" stroke="#f1f5f9" stroke-width="1"/>

  <!-- Total -->
  <text x="592" y="364" text-anchor="middle" fill="#94a3b8" font-size="10">Total en disco</text>
  <text x="592" y="396" text-anchor="middle" fill="#1e3a5f" font-size="30" font-weight="bold">~ 250 GB</text>
  <text x="592" y="416" text-anchor="middle" fill="#94a3b8" font-size="9.5">9 datasets · 45 keyframes</text>
  <text x="592" y="430" text-anchor="middle" fill="#94a3b8" font-size="9.5">3 cerdos distintos [README]</text>
  <text x="592" y="446" text-anchor="middle" fill="#94a3b8" font-size="9">MICCAI 2019 · versión extendida</text>
  <text x="592" y="460" text-anchor="middle" fill="#94a3b8" font-size="9">HuggingFace (original: 3 datasets)</text>

  <!-- ════════════════════════════════
       NOTA AL PIE
       ════════════════════════════════ -->
  <rect x="18" y="494" width="724" height="108" rx="8"
        fill="#f8fafc" stroke="#e2e8f0" stroke-width="1.2"/>

  <text x="380" y="514" text-anchor="middle" fill="#1e3a5f" font-size="11" font-weight="bold">
    Código de color
  </text>
  <line x1="36" y1="521" x2="724" y2="521" stroke="#e2e8f0" stroke-width="1"/>

  <rect x="50"  y="530" width="5" height="20" rx="1" fill="#16a34a"/>
  <text x="64"  y="545" fill="#334155" font-size="10.5" font-weight="bold">dataset_1–7</text>
  <text x="64"  y="560" fill="#64748b" font-size="9.5">keyframe_1…5</text>

  <rect x="220" y="530" width="5" height="20" rx="1" fill="#0891b2"/>
  <text x="234" y="545" fill="#334155" font-size="10.5" font-weight="bold">dataset_8–9</text>
  <text x="234" y="560" fill="#64748b" font-size="9.5">keyframe_0…4</text>

  <rect x="390" y="530" width="5" height="20" rx="1" fill="#7c3aed"/>
  <text x="404" y="545" fill="#334155" font-size="10.5" font-weight="bold">test_dataset</text>
  <text x="404" y="560" fill="#64748b" font-size="9.5">Sin scene_points</text>

  <rect x="560" y="530" width="5" height="20" rx="1" fill="#64748b"/>
  <text x="574" y="545" fill="#334155" font-size="10.5" font-weight="bold">Auxiliares</text>
  <text x="574" y="560" fill="#64748b" font-size="9.5">README · scripts</text>

  <text x="380" y="586" text-anchor="middle" fill="#94a3b8" font-size="9">
    Fuente: SCARED — Stereo Correspondence and Reconstruction of Endoscopic Data · MICCAI 2019
  </text>

</svg>
"""

display(SVG(svg1))
print("💾  Clic derecho → 'Guardar imagen como...' para exportar a Word")

In [ ]:
svg2 = """
<svg width="960" height="780" xmlns="http://www.w3.org/2000/svg"
     font-family="'Segoe UI', Arial, sans-serif">

  <!-- Fondo blanco -->
  <rect width="960" height="780" rx="12" fill="#ffffff" stroke="#e2e8f0" stroke-width="1.5"/>

  <!-- Banda de título -->
  <rect x="0" y="0" width="960" height="50" rx="12" fill="#1e3a5f"/>
  <rect x="0" y="38" width="960" height="12" fill="#1e3a5f"/>
  <text x="480" y="30" text-anchor="middle" fill="#ffffff" font-size="15" font-weight="bold">
    Estructura interna de dataset_N.zip — contenido de cada keyframe
  </text>
  <text x="480" y="60" text-anchor="middle" fill="#64748b" font-size="11">
    Todos los keyframes tienen la misma estructura interna (excepciones indicadas)
  </text>

  <!-- ═══════════════════════════════════════
       COLUMNA IZQUIERDA: lista de keyframes
       ═══════════════════════════════════════ -->

  <rect x="20" y="78" width="192" height="30" rx="6"
        fill="none" stroke="#1e3a5f" stroke-width="1.5"/>
  <text x="32" y="98" fill="#1e3a5f" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">dataset_N.zip</text>

  <line x1="46" y1="108" x2="46" y2="430" stroke="#cbd5e1" stroke-width="1.5"/>

  <!-- keyframe_1 resaltado -->
  <line x1="46" y1="134" x2="70" y2="134" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="70" y="120" width="142" height="28" rx="5"
        fill="#eff6ff" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="82" y="138" fill="#1d4ed8" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">keyframe_1/</text>

  <!-- keyframe_2 -->
  <line x1="46" y1="172" x2="70" y2="172" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="70" y="158" width="142" height="28" rx="5"
        fill="none" stroke="#93c5fd" stroke-width="1.2"/>
  <text x="82" y="176" fill="#3b82f6" font-size="12"
        font-family="'Courier New', monospace">keyframe_2/</text>

  <!-- keyframe_3 -->
  <line x1="46" y1="210" x2="70" y2="210" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="70" y="196" width="142" height="28" rx="5"
        fill="none" stroke="#93c5fd" stroke-width="1.2"/>
  <text x="82" y="214" fill="#3b82f6" font-size="12"
        font-family="'Courier New', monospace">keyframe_3/</text>

  <!-- keyframe_4 excepción -->
  <line x1="46" y1="248" x2="70" y2="248" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="70" y="234" width="142" height="28" rx="5"
        fill="#fff7ed" stroke="#f97316" stroke-width="1.2" stroke-dasharray="5,3"/>
  <text x="82" y="252" fill="#c2410c" font-size="12"
        font-family="'Courier New', monospace">keyframe_4/ ⚠</text>

  <!-- keyframe_5 -->
  <line x1="46" y1="286" x2="70" y2="286" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="70" y="272" width="142" height="28" rx="5"
        fill="none" stroke="#93c5fd" stroke-width="1.2"/>
  <text x="82" y="290" fill="#3b82f6" font-size="12"
        font-family="'Courier New', monospace">keyframe_5/</text>

  <!-- Notas -->
  <rect x="20" y="318" width="212" height="52" rx="6"
        fill="#fff7ed" stroke="#fed7aa" stroke-width="1"/>
  <text x="30" y="335" fill="#c2410c" font-size="10.5" font-weight="bold">⚠  keyframe_4 / dataset_1</text>
  <text x="30" y="350" fill="#78350f" font-size="10">Sin carpeta data/ — error de</text>
  <text x="30" y="363" fill="#78350f" font-size="10">logging en la captura [README]</text>

  <rect x="20" y="378" width="212" height="52" rx="6"
        fill="#f0f9ff" stroke="#bae6fd" stroke-width="1"/>
  <text x="30" y="395" fill="#0369a1" font-size="10.5" font-weight="bold">ℹ  keyframe_5 (todos)</text>
  <text x="30" y="410" fill="#075985" font-size="10">Sin data/ en datasets originales</text>
  <text x="30" y="423" fill="#075985" font-size="10">del challenge (último keyframe)</text>

  <!-- Flecha -->
  <defs>
    <marker id="arr2" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#3b82f6"/>
    </marker>
  </defs>
  <line x1="212" y1="134" x2="254" y2="134" stroke="#3b82f6" stroke-width="1.5"
        stroke-dasharray="5,3" marker-end="url(#arr2)"/>

  <!-- ═══════════════════════════════════════
       COLUMNA DERECHA: archivos
       Cada fila: alto=44px  nombre en y+16  descripción en y+32
       ═══════════════════════════════════════ -->

  <rect x="254" y="78" width="686" height="30" rx="6"
        fill="#eff6ff" stroke="#3b82f6" stroke-width="1.5"/>
  <text x="268" y="98" fill="#1d4ed8" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">keyframe_N/  — archivos</text>

  <line x1="282" y1="108" x2="282" y2="600" stroke="#cbd5e1" stroke-width="1.5"/>

  <!-- ── Left_Image.png  (azul) ── y_top=118 -->
  <line x1="282" y1="140" x2="306" y2="140" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="118" width="628" height="44" rx="5" fill="none" stroke="#bfdbfe" stroke-width="1"/>
  <rect x="306" y="118" width="5"   height="44" rx="2" fill="#2563eb"/>
  <text x="320" y="136" fill="#1e40af" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">Left_Image.png</text>
  <text x="320" y="153" fill="#64748b" font-size="10.5">2.5 MB · RGB uint8 · 1280×1024 px · entrada del modelo</text>

  <!-- ── Right_Image.png  (azul) ── y_top=172 -->
  <line x1="282" y1="194" x2="306" y2="194" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="172" width="628" height="44" rx="5" fill="none" stroke="#bfdbfe" stroke-width="1"/>
  <rect x="306" y="172" width="5"   height="44" rx="2" fill="#2563eb"/>
  <text x="320" y="190" fill="#1e40af" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">Right_Image.png</text>
  <text x="320" y="207" fill="#64748b" font-size="10.5">2.5 MB · RGB uint8 · cámara derecha (par estéreo)</text>

  <!-- ── left_depth_map.tiff  (naranja) ── y_top=226 -->
  <line x1="282" y1="248" x2="306" y2="248" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="226" width="628" height="44" rx="5" fill="none" stroke="#fed7aa" stroke-width="1"/>
  <rect x="306" y="226" width="5"   height="44" rx="2" fill="#ea580c"/>
  <text x="320" y="244" fill="#9a3412" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">left_depth_map.tiff</text>
  <text x="320" y="261" fill="#64748b" font-size="10.5">15.7 MB · (X,Y,Z) float32 · mm · GT supervisión</text>

  <!-- ── right_depth_map.tiff  (naranja) ── y_top=280 -->
  <line x1="282" y1="302" x2="306" y2="302" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="280" width="628" height="44" rx="5" fill="none" stroke="#fed7aa" stroke-width="1"/>
  <rect x="306" y="280" width="5"   height="44" rx="2" fill="#ea580c"/>
  <text x="320" y="298" fill="#9a3412" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">right_depth_map.tiff</text>
  <text x="320" y="315" fill="#64748b" font-size="10.5">15.7 MB · (X,Y,Z) float32 · mm · cámara derecha</text>

  <!-- ── point_cloud.obj  (verde) ── y_top=334 -->
  <line x1="282" y1="356" x2="306" y2="356" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="334" width="628" height="44" rx="5" fill="none" stroke="#bbf7d0" stroke-width="1"/>
  <rect x="306" y="334" width="5"   height="44" rx="2" fill="#16a34a"/>
  <text x="320" y="352" fill="#15803d" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">point_cloud.obj</text>
  <text x="320" y="369" fill="#64748b" font-size="10.5">63.6 MB · vértices 3D · GT absoluto (Chamfer Distance)</text>

  <!-- ── endoscope_calibration.yaml  (gris) ── y_top=388 -->
  <line x1="282" y1="410" x2="306" y2="410" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="388" width="628" height="44" rx="5" fill="none" stroke="#cbd5e1" stroke-width="1"/>
  <rect x="306" y="388" width="5"   height="44" rx="2" fill="#64748b"/>
  <text x="320" y="406" fill="#334155" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">endoscope_calibration.yaml</text>
  <text x="320" y="423" fill="#64748b" font-size="10.5">~0 MB · matriz K intrínseca · aproximada [README]</text>

  <!-- ── data/  carpeta ── y_top=442 -->
  <line x1="282" y1="458" x2="306" y2="458" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="306" y="444" width="108" height="28" rx="5"
        fill="#f0fdf4" stroke="#86efac" stroke-width="1.2"/>
  <text x="318" y="462" fill="#15803d" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">data/</text>

  <line x1="334" y1="472" x2="334" y2="600" stroke="#cbd5e1" stroke-width="1.5" stroke-dasharray="3,3"/>

  <!-- ── rgb.mp4  (morado) ── y_top=482 -->
  <line x1="334" y1="504" x2="358" y2="504" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="358" y="482" width="576" height="44" rx="5" fill="none" stroke="#e9d5ff" stroke-width="1"/>
  <rect x="358" y="482" width="5"   height="44" rx="2" fill="#7c3aed"/>
  <text x="372" y="500" fill="#5b21b6" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">rgb.mp4</text>
  <text x="372" y="517" fill="#64748b" font-size="10.5">33 MB · video L+R apilado verticalmente · solo OpenCV/ffmpeg [README]</text>

  <!-- ── frame_data.tar.gz  (morado) ── y_top=536 -->
  <line x1="334" y1="558" x2="358" y2="558" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="358" y="536" width="576" height="44" rx="5" fill="none" stroke="#e9d5ff" stroke-width="1"/>
  <rect x="358" y="536" width="5"   height="44" rx="2" fill="#7c3aed"/>
  <text x="372" y="554" fill="#5b21b6" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">frame_data.tar.gz</text>
  <text x="372" y="571" fill="#64748b" font-size="10.5">~0 MB · JSONs frame_data%06d · pose de cámara por frame</text>

  <!-- ── scene_points.tar.gz  (naranja advertencia) ── y_top=590 -->
  <line x1="334" y1="612" x2="358" y2="612" stroke="#cbd5e1" stroke-width="1.5"/>
  <rect x="358" y="590" width="576" height="44" rx="5" fill="#fff7ed" stroke="#f97316" stroke-width="1.2"/>
  <rect x="358" y="590" width="5"   height="44" rx="2" fill="#ea580c"/>
  <text x="372" y="608" fill="#9a3412" font-size="12" font-weight="bold"
        font-family="'Courier New', monospace">scene_points.tar.gz</text>
  <text x="372" y="625" fill="#9a3412" font-size="10.5">2 611 MB · TIFFs warpeados · ⚠ no es GT perfecto [README]</text>

  <!-- ═══════════════════════════
       LEYENDA INFERIOR
       ═══════════════════════════ -->
  <rect x="20" y="652" width="920" height="110" rx="8"
        fill="#f8fafc" stroke="#e2e8f0" stroke-width="1.2"/>

  <text x="480" y="672" text-anchor="middle" fill="#1e3a5f" font-size="12" font-weight="bold">
    Código de color por tipo de archivo
  </text>
  <line x1="40" y1="679" x2="900" y2="679" stroke="#e2e8f0" stroke-width="1"/>

  <rect x="50"  y="688" width="5" height="20" rx="1" fill="#2563eb"/>
  <text x="64"  y="702" fill="#334155" font-size="11" font-weight="bold">PNG</text>
  <text x="64"  y="716" fill="#64748b" font-size="10">Imagen RGB</text>

  <rect x="200" y="688" width="5" height="20" rx="1" fill="#ea580c"/>
  <text x="214" y="702" fill="#334155" font-size="11" font-weight="bold">TIFF</text>
  <text x="214" y="716" fill="#64748b" font-size="10">Depth XYZ float32</text>

  <rect x="360" y="688" width="5" height="20" rx="1" fill="#16a34a"/>
  <text x="374" y="702" fill="#334155" font-size="11" font-weight="bold">OBJ</text>
  <text x="374" y="716" fill="#64748b" font-size="10">Nube de puntos 3D</text>

  <rect x="510" y="688" width="5" height="20" rx="1" fill="#64748b"/>
  <text x="524" y="702" fill="#334155" font-size="11" font-weight="bold">YAML</text>
  <text x="524" y="716" fill="#64748b" font-size="10">Calibración intrínseca (aprox.)</text>

  <rect x="700" y="688" width="5" height="20" rx="1" fill="#7c3aed"/>
  <text x="714" y="702" fill="#334155" font-size="11" font-weight="bold">MP4 / TAR.GZ</text>
  <text x="714" y="716" fill="#64748b" font-size="10">Video y archivos empaquetados</text>

  <!-- Nota scene_points -->
  <rect x="40" y="728" width="880" height="26" rx="5"
        fill="#fff7ed" stroke="#fed7aa" stroke-width="1"/>
  <text x="54" y="745" fill="#92400e" font-size="10.5" font-weight="bold">⚠  scene_points.tar.gz:</text>
  <text x="208" y="745" fill="#78350f" font-size="10.5">
    depth warpeado NO es GT perfecto — errores por cinemática o sincronización [README]. Usar solo para entrenamiento auxiliar.
  </text>

</svg>
"""

display(SVG(svg2))
print("💾  Clic derecho → 'Guardar imagen como...' para exportar a Word")

---
## Sección 1 — Configuración e imports

Primero instalamos lo que podría faltar y cargamos las librerías.

In [ ]:
!pip install tifffile --quiet

In [ ]:
import zipfile
import io
import tarfile
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml

# ── Cambia esta ruta si tus ZIPs están en otro lugar ─────────────────────────
SCARED_DIR = Path("E:/scared_raw")
# ─────────────────────────────────────────────────────────────────────────────

# Verifica que la carpeta existe
assert SCARED_DIR.exists(), f"No encuentro la carpeta: {SCARED_DIR}"

# Lista todos los ZIPs de dataset (excluye los test_* que son parciales)
dataset_zips = sorted(SCARED_DIR.glob("dataset_*.zip"))
print(f"Carpeta encontrada: {SCARED_DIR}")
print(f"ZIPs de dataset encontrados: {len(dataset_zips)}")
for z in dataset_zips:
    print(f"  {z.name:25s}  {z.stat().st_size / 1e9:.1f} GB")

---
## Sección 2 — ¿Qué hay dentro de los ZIPs?

Cada ZIP contiene un dataset con varios **keyframes**. Un keyframe es un momento concreto en el que se iluminó el tejido con luz estructurada y se capturó todo: la imagen RGB de la cámara izquierda, el mapa de profundidad, la nube de puntos, el video del movimiento posterior, etc.

La celda siguiente abre **solo el índice** de cada ZIP (sin descomprimir nada al disco) y construye un mapa de su contenido.

In [ ]:
def describe_zip(zip_path):
    """
    Lee solo el índice del ZIP (no descomprime) y devuelve un dict con:
      - keyframes: lista de keyframes encontrados
      - files_per_kf: {kf_name: [lista de archivos y sus tamaños en MB]}
      - extensions: Counter de extensiones en todo el ZIP
    """
    kf_pattern = re.compile(r"dataset_\d+/(keyframe_\d+)/(.+)")
    kf_files = defaultdict(list)
    ext_sizes = defaultdict(float)

    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            m = kf_pattern.match(info.filename)
            if not m:
                continue
            kf_name, rel_path = m.group(1), m.group(2)
            size_mb = info.file_size / 1e6
            kf_files[kf_name].append((rel_path, size_mb))
            ext = Path(rel_path).suffix.lower() or "(sin extensión)"
            ext_sizes[ext] += size_mb

    return {
        "keyframes": sorted(kf_files.keys()),
        "files_per_kf": dict(kf_files),
        "ext_sizes_mb": dict(ext_sizes),
    }

# Describimos cada ZIP
catalog = {}
for z in dataset_zips:
    print(f"Leyendo índice de {z.name} ...", end=" ")
    catalog[z.stem] = describe_zip(z)
    kfs = catalog[z.stem]["keyframes"]
    print(f"{len(kfs)} keyframes ({', '.join(kfs)})")

print("\nListo.")

### 2.1 — Árbol de archivos de un keyframe

Veamos exactamente qué archivos hay en `dataset_1 / keyframe_1`. Esto es representativo de todos los demás keyframes.

In [ ]:
TARGET_DATASET = "dataset_1"
TARGET_KF      = "keyframe_1"

files = catalog[TARGET_DATASET]["files_per_kf"][TARGET_KF]
files_sorted = sorted(files, key=lambda x: x[0])

print(f"Archivos en {TARGET_DATASET}/{TARGET_KF}:")
print(f"{'Ruta relativa':<45} {'Tamaño':>10}")
print("-" * 57)
for rel_path, size_mb in files_sorted:
    print(f"  {rel_path:<43} {size_mb:>8.1f} MB")

print(f"\nTotal: {len(files_sorted)} archivos")

**¿Qué es cada archivo?**

| Archivo | Formato | Qué contiene y para qué sirve |
|---|---|---|
| `Left_Image.png` | PNG · uint8 · 1280×1024 px | Foto a color de la cámara izquierda capturada en el instante exacto en que el proyector de luz estructurada iluminó el tejido. Es la **entrada principal del modelo** de corrección de iluminación: contiene la textura real del tejido más los gradientes de luz no uniforme que queremos corregir. |
| `Right_Image.png` | PNG · uint8 · 1280×1024 px | Misma captura desde la cámara derecha. Junto con `Left_Image.png` forma el **par estéreo**: dos vistas del mismo escenario desde ángulos ligeramente distintos, lo que permite calcular profundidad por disparidad si se necesita un método alternativo al structured light. |
| `left_depth_map.tiff` | TIFF · float32 · (H, W, 3) | **Ground Truth de profundidad** de la cámara izquierda. Cada píxel almacena una tripleta `(X, Y, Z)` en milímetros que indica la posición 3D del punto del tejido que corresponde a ese píxel. Los píxeles donde el structured light no pudo medir (reflejos especulares, zonas muy curvadas) contienen `(0, 0, 0)`. Se lee con `tifffile`, no con visores de imagen normales. |
| `right_depth_map.tiff` | TIFF · float32 · (H, W, 3) | Mismo formato que `left_depth_map.tiff` pero desde el punto de vista de la cámara derecha. Útil para validación cruzada o para modelos que consumen el par estéreo completo. |
| `point_cloud.obj` | Wavefront OBJ · texto | Nube de puntos 3D capturada **directamente por el proyector de luz estructurada** antes de cualquier interpolación. Es la referencia más fiel de la geometría del tejido y se usa como GT para métricas como la Chamfer Distance. Cada línea `v x y z` es un vértice en mm. Pesa ~60 MB; se lee parseando líneas `v ` sin cargar librerías 3D completas. |
| `endoscope_calibration.yaml` | YAML · formato OpenCV | Parámetros intrínsecos de la cámara: **matriz K** (focal `fx`, `fy` y centro óptico `cx`, `cy`) y coeficientes de distorsión. Permite convertir coordenadas de píxel a rayos 3D (reproyección). El README advierte que es una calibración **aproximada** — los modelos deben ser robustos a pequeños errores en K. Se carga con `cv2.FileStorage`. |
| `data/rgb.mp4` | MP4 · H.264 | Secuencia de video grabada **después** de la captura del keyframe, mientras el endoscopio se mueve libremente por el tejido. Los frames tienen ambas cámaras **apiladas verticalmente** (izquierda arriba, derecha abajo). No abre en VLC ni QuickTime — requiere OpenCV o ffmpeg. Sirve para extraer pares imagen–profundidad adicionales mediante warping. |
| `data/frame_data.tar.gz` | TAR.GZ → JSONs | Archivo comprimido con un JSON por cada frame del video (`frame_data000001.json`, etc.). Cada JSON contiene la **transformación rígida de la cámara** (rotación + traslación) relativa a la posición en que se capturó el keyframe, más los parámetros de calibración de ese frame. Necesario para warpear el depth map del keyframe a la posición de cada frame del video. |
| `data/scene_points.tar.gz` | TAR.GZ → TIFFs · float32 | El archivo más grande del dataset (2–6 GB por keyframe). Contiene un TIFF `(X, Y, Z)` por cada frame del video con la nube de puntos del keyframe **proyectada y warpeada** a la posición de cámara de ese frame. Los píxeles que quedan fuera del campo de visión del keyframe contienen `(0, 0, 0)`. El README advierte que el warping no es GT perfecto: pueden existir pequeños errores por la cinemática del robot o desincronización entre cámara y video. Usar solo para entrenamiento auxiliar, no para métricas finales. |

> **Nota sobre `keyframe_4` de `dataset_1`:** le falta la carpeta `data/` completa por un error de logging durante la captura original. Los cinco archivos estáticos (imágenes, TIFFs, OBJ, YAML) sí están presentes.

### 2.2 — Resumen de todos los datasets

¿Todos los datasets tienen la misma estructura? ¿Cuántos keyframes tiene cada uno?

In [ ]:
print(f"{'Dataset':<12} {'Keyframes':<50} {'N':<4} {'ZIP (GB)'}")
print("-" * 75)
for z in dataset_zips:
    info = catalog[z.stem]
    kfs  = info["keyframes"]
    size_gb = z.stat().st_size / 1e9
    print(f"  {z.stem:<10} {', '.join(kfs):<50} {len(kfs):<4} {size_gb:.1f} GB")

total_gb = sum(z.stat().st_size for z in dataset_zips) / 1e9
print(f"\nTotal: {len(dataset_zips)} datasets, {total_gb:.0f} GB")
print()
print("⚠  ATENCIÓN: dataset_8 y dataset_9 usan keyframe_0…keyframe_4 (índice desde 0).")
print("   Los demás usan keyframe_1…keyframe_5 (índice desde 1).")
print("   No hardcodear ['keyframe_1',...,'keyframe_5'] — usar catalog[ds]['keyframes'].")

---
## Sección 3 — Funciones de lectura en memoria

Para no extraer nada al disco, todas las funciones siguientes leen directamente desde el ZIP usando `io.BytesIO`.  
Son las mismas que usaremos en todos los notebooks de este proyecto.

In [ ]:
def read_image(zf, dataset, kf, side="Left"):
    """
    Lee Left_Image.png o Right_Image.png y devuelve array RGB uint8 (H, W, 3).
    - zf: objeto ZipFile abierto
    - side: "Left" o "Right"
    """
    path = f"{dataset}/{kf}/{side}_Image.png"
    buf  = np.frombuffer(zf.read(path), np.uint8)
    img  = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def read_depth(zf, dataset, kf, side="left"):
    """
    Lee left_depth_map.tiff o right_depth_map.tiff.
    El TIFF almacena (X, Y, Z) por píxel; nosotros extraemos solo Z (profundidad en mm).
    Devuelve array float32 (H, W). Los píxeles sin medición válida son NaN.
    """
    path = f"{dataset}/{kf}/{side}_depth_map.tiff"
    with zf.open(path) as f:
        data = tifffile.imread(io.BytesIO(f.read()))
    # Si el TIFF tiene 3 canales (X,Y,Z), tomamos el canal Z (índice 2)
    z = data[..., 2].astype(np.float32) if (data.ndim == 3 and data.shape[2] >= 3) else data.astype(np.float32)
    z[z <= 0] = np.nan   # valores ≤ 0 son píxeles sin medición
    return z


def read_calibration(zf, dataset, kf):
    """
    Lee endoscope_calibration.yaml y devuelve la matriz intrínseca K (3x3, float32).
    K = [[fx, 0, cx],
         [0, fy, cy],
         [0,  0,  1]]
    """
    import tempfile, os
    path = f"{dataset}/{kf}/endoscope_calibration.yaml"
    with zf.open(path) as f:
        content = f.read()
    # OpenCV usa un formato YAML especial; necesitamos escribirlo a disco temporalmente
    with tempfile.NamedTemporaryFile(suffix=".yaml", delete=False) as tmp:
        tmp.write(content)
        tmp_path = tmp.name
    fs = cv2.FileStorage(tmp_path, cv2.FILE_STORAGE_READ)
    K  = fs.getNode("M1").mat()
    fs.release()
    os.unlink(tmp_path)
    return K.astype(np.float32) if K is not None else None


def read_obj_sample(zf, dataset, kf, max_pts=5000):
    """
    Lee point_cloud.obj y devuelve hasta max_pts vértices como array (N, 3) float32.
    Solo lee las líneas que empiezan con 'v ' (vértices XYZ).
    """
    path = f"{dataset}/{kf}/point_cloud.obj"
    lines = zf.read(path).decode("utf-8", errors="ignore").splitlines()
    pts = [[float(p) for p in l.split()[1:4]] for l in lines if l.startswith("v ")]
    arr = np.array(pts, np.float32)
    if len(arr) > max_pts:
        idx = np.random.choice(len(arr), max_pts, replace=False)
        arr = arr[idx]
    return arr


print("Funciones de lectura definidas: read_image, read_depth, read_calibration, read_obj_sample")

---
## Sección 4 — PNG: imagen RGB (`Left_Image.png`)

La imagen más importante para nuestro proyecto: es la foto del tejido que el modelo recibe como entrada. Vemos la izquierda y la derecha para entender el par estéreo.

In [ ]:
zip_path = SCARED_DIR / "dataset_1.zip"

with zipfile.ZipFile(zip_path) as zf:
    img_left  = read_image(zf, "dataset_1", "keyframe_1", side="Left")
    img_right = read_image(zf, "dataset_1", "keyframe_1", side="Right")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(img_left)
axes[0].set_title("Left_Image.png — cámara izquierda", fontsize=13)
axes[0].axis("off")
axes[1].imshow(img_right)
axes[1].set_title("Right_Image.png — cámara derecha", fontsize=13)
axes[1].axis("off")
plt.suptitle("dataset_1 / keyframe_1 — par estéreo", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Propiedades del array
print(f"Forma del array:  {img_left.shape}  → (alto, ancho, canales)")
print(f"Tipo de dato:     {img_left.dtype}  → valores de 0 a 255")
print(f"Rango de valores: [{img_left.min()}, {img_left.max()}]")
print(f"Resolución:       {img_left.shape[1]}×{img_left.shape[0]} px")

---
## Sección 5 — TIFF: mapa de profundidad (`left_depth_map.tiff`)

Este archivo es el Ground Truth (GT) de profundidad. **No es una imagen normal**: cada píxel guarda las coordenadas 3D `(X, Y, Z)` en milímetros del punto del tejido que le corresponde. Nosotros usamos solo el canal Z (la distancia al tejido).

Los píxeles donde el structured light no pudo medir (reflejos especulares, bordes muy curvos) tienen valor `(0, 0, 0)` — los convertimos a `NaN`.

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    # Leer el TIFF "crudo" para ver todos sus canales
    with zf.open("dataset_1/keyframe_1/left_depth_map.tiff") as f:
        raw_tiff = tifffile.imread(io.BytesIO(f.read()))

print("=== TIFF crudo ===")
print(f"Forma:     {raw_tiff.shape}  → (alto, ancho, canales XYZ)")
print(f"Tipo:      {raw_tiff.dtype}")
print(f"Rango X:   [{np.nanmin(raw_tiff[...,0]):.1f}, {np.nanmax(raw_tiff[...,0]):.1f}] mm")
print(f"Rango Y:   [{np.nanmin(raw_tiff[...,1]):.1f}, {np.nanmax(raw_tiff[...,1]):.1f}] mm")
print(f"Rango Z:   [{np.nanmin(raw_tiff[...,2]):.1f}, {np.nanmax(raw_tiff[...,2]):.1f}] mm")

# Extraer solo Z y crear la máscara de validez
depth = raw_tiff[..., 2].astype(np.float32)
valid = depth > 0
depth_clean = depth.copy()
depth_clean[~valid] = np.nan

print(f"\n=== Canal Z (profundidad) ===")
print(f"Píxeles válidos:  {valid.sum():,} de {valid.size:,}  ({valid.mean()*100:.1f}%)")
print(f"Profundidad mín:  {np.nanmin(depth_clean):.1f} mm")
print(f"Profundidad med:  {np.nanmedian(depth_clean):.1f} mm")
print(f"Profundidad máx:  {np.nanmax(depth_clean):.1f} mm")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Canal Z como mapa de calor
im = axes[0].imshow(depth_clean, cmap="viridis")
axes[0].set_title("Canal Z — profundidad en mm\n(amarillo=cercano, morado=lejano)", fontsize=11)
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046, label="mm")

# Máscara de validez: verde=tiene GT, rojo=sin GT
axes[1].imshow(valid.astype(np.float32), cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title(f"Máscara de validez\n(verde = tiene GT, rojo = sin medición)", fontsize=11)
axes[1].axis("off")

# Histograma de profundidades válidas
axes[2].hist(depth_clean[valid], bins=100, color="#2a9d8f", edgecolor="none")
axes[2].set_xlabel("Profundidad (mm)")
axes[2].set_ylabel("Cantidad de píxeles")
axes[2].set_title("Distribución de profundidades válidas", fontsize=11)
axes[2].axvline(np.nanmedian(depth_clean), color="red", linestyle="--", label=f"Mediana: {np.nanmedian(depth_clean):.0f} mm")
axes[2].legend()

plt.suptitle("left_depth_map.tiff — dataset_1 / keyframe_1", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Sección 6 — YAML: calibración de la cámara (`endoscope_calibration.yaml`)

La calibración nos dice las propiedades ópticas de la cámara: distancia focal y punto principal. Con esta información podemos convertir píxeles a coordenadas 3D reales (reproyección).

La matriz `K` tiene esta forma:
```
K = [[fx,  0, cx],
     [ 0, fy, cy],
     [ 0,  0,  1]]
```
- `fx`, `fy`: longitud focal en píxeles
- `cx`, `cy`: coordenadas del punto principal (centro óptico proyectado al sensor)

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    # Ver el contenido crudo del YAML primero
    with zf.open("dataset_1/keyframe_1/endoscope_calibration.yaml") as f:
        raw_yaml = f.read().decode("utf-8", errors="ignore")

print("=== Contenido crudo del YAML ===")
print(raw_yaml[:800])  # primeros 800 caracteres

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    K = read_calibration(zf, "dataset_1", "keyframe_1")

if K is not None:
    print("=== Matriz intrínseca K (cámara izquierda) ===")
    print(f"\n  fx = {K[0,0]:.1f} px   (focal horizontal)")
    print(f"  fy = {K[1,1]:.1f} px   (focal vertical)")
    print(f"  cx = {K[0,2]:.1f} px   (centro óptico, eje X)")
    print(f"  cy = {K[1,2]:.1f} px   (centro óptico, eje Y)")
    print(f"\nMatriz completa:\n{K}")
    H, W = img_left.shape[:2]
    print(f"\nResolución de la imagen: {W}×{H} px")
    print(f"Centro teórico del sensor: ({W/2:.0f}, {H/2:.0f}) px")
    print(f"Centro óptico real:        ({K[0,2]:.1f}, {K[1,2]:.1f}) px")
else:
    print("No se pudo leer la calibración.")

---
## Sección 7 — OBJ: nube de puntos (`point_cloud.obj`)

El archivo `.obj` es la nube de puntos 3D capturada **directamente por structured light** — sin ningún procesamiento intermedio. Es la referencia más confiable de la geometría real del tejido.

El archivo es muy grande (60–70 MB), así que solo leemos una muestra de vértices para visualizar la forma.

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    # Ver las primeras líneas del OBJ para entender el formato
    raw_obj = zf.read("dataset_1/keyframe_1/point_cloud.obj").decode("utf-8", errors="ignore")

lines = raw_obj.splitlines()
print("=== Primeras 15 líneas del archivo OBJ ===")
for line in lines[:15]:
    print(f"  {line}")

# Contar vértices y caras
n_vertices = sum(1 for l in lines if l.startswith("v "))
n_faces    = sum(1 for l in lines if l.startswith("f "))
print(f"\nTotal de vértices (líneas 'v '): {n_vertices:,}")
print(f"Total de caras    (líneas 'f '): {n_faces:,}")

In [ ]:
# Reutiliza `lines` de la celda anterior si está disponible;
# si no (ejecución fuera de orden), relee el OBJ del ZIP.
if "lines" not in dir() or not isinstance(lines, list) or len(lines) == 0:
    with zipfile.ZipFile(zip_path) as zf:
        lines = zf.read("dataset_1/keyframe_1/point_cloud.obj") \
                  .decode("utf-8", errors="ignore").splitlines()

pts_all = []
for line in lines:
    if not line.startswith("v "):
        continue
    parts = line.split()
    if len(parts) < 4:
        continue
    try:
        pts_all.append([float(parts[1]), float(parts[2]), float(parts[3])])
    except ValueError:
        continue

if len(pts_all) == 0:
    raise RuntimeError(
        f"No se encontraron vértices válidos. "
        f"Primeras 5 líneas del OBJ: {lines[:5]}"
    )

pts_all = np.array(pts_all, dtype=np.float32)

# Filtra filas con NaN o Inf (coordenadas corruptas)
mask = np.isfinite(pts_all).all(axis=1)
pts_all = pts_all[mask]

# Muestra aleatoria reproducible
rng = np.random.default_rng(42)
if len(pts_all) > 8000:
    idx = rng.choice(len(pts_all), 8000, replace=False)
    pts = pts_all[idx]
else:
    pts = pts_all

print(f"Muestra cargada: {len(pts):,} puntos de {len(pts_all):,} totales")
print(f"Rango X: [{pts[:,0].min():.1f}, {pts[:,0].max():.1f}] mm")
print(f"Rango Y: [{pts[:,1].min():.1f}, {pts[:,1].max():.1f}] mm")
print(f"Rango Z: [{pts[:,2].min():.1f}, {pts[:,2].max():.1f}] mm")

# Visualización 2D de la nube (proyección XZ y XY)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(pts[:,0], pts[:,2], c=pts[:,2], cmap="viridis", s=0.5, alpha=0.6)
axes[0].set_xlabel("X (mm)")
axes[0].set_ylabel("Z — profundidad (mm)")
axes[0].set_title("Proyección XZ\n(vista lateral: qué tan lejos está cada punto)")
plt.colorbar(sc, ax=axes[0], label="Z mm")

axes[1].scatter(pts[:,0], pts[:,1], c=pts[:,2], cmap="viridis", s=0.5, alpha=0.6)
axes[1].set_xlabel("X (mm)")
axes[1].set_ylabel("Y (mm)")
axes[1].set_title("Proyección XY\n(vista frontal: forma del tejido)")
axes[1].invert_yaxis()

plt.suptitle("point_cloud.obj — dataset_1 / keyframe_1 (muestra de 8 000 pts)", fontsize=13)
plt.tight_layout()
plt.show()

---
## Sección 8 — TAR.GZ: `frame_data.tar.gz` (poses de cámara)

Dentro de `data/frame_data.tar.gz` hay cientos de archivos JSON, uno por cuadro del video. Cada JSON describe la posición y orientación de la cámara en ese momento, relativa a donde estaba cuando se tomó el keyframe.

**¿Para qué sirve?** Para saber cómo se movió el endoscopio entre el keyframe y cada cuadro del video — información necesaria para warpear la profundidad al nuevo punto de vista.

Aquí solo abrimos el tar y mostramos la cabeza de uno de los JSONs.

In [ ]:
import json

with zipfile.ZipFile(zip_path) as zf:
    tar_bytes = io.BytesIO(zf.read("dataset_1/keyframe_1/data/frame_data.tar.gz"))

with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
    members = tar.getmembers()
    print(f"Archivos dentro del tar: {len(members)}")
    print(f"Primeros 5: {[m.name for m in members[:5]]}")
    print(f"Últimos  5: {[m.name for m in members[-5:]]}")

    # Leer el primer JSON
    first_json_member = next(m for m in members if m.name.endswith(".json"))
    f_json = tar.extractfile(first_json_member)
    data_json = json.load(f_json)

print(f"\n=== Estructura del JSON ({first_json_member.name}) ===")
print(json.dumps(data_json, indent=2)[:1000])

---
## Sección 9 — TAR.GZ: `scene_points.tar.gz` (profundidad warpeada por cuadro)

Este es el archivo más grande del dataset (2–6 GB por keyframe). Contiene un TIFF de profundidad por cada cuadro del video — la geometría del keyframe proyectada a la posición de la cámara en ese cuadro.

**No vamos a leer todo el tar** (sería lentísimo). Solo abrimos el índice para saber cuántos frames hay, y luego leemos un solo TIFF de muestra.

In [ ]:
# ADVERTENCIA: leer el índice del scene_points.tar.gz puede tardar 1-3 minutos
# porque el tar está comprimido y hay que descomprimirlo en streaming para leer el índice.
# Si no necesitas este análisis ahora mismo, puedes saltar esta celda.

print("Leyendo índice de scene_points.tar.gz (puede tardar ~1-2 minutos)...")

with zipfile.ZipFile(zip_path) as zf:
    sp_bytes = io.BytesIO(zf.read("dataset_1/keyframe_1/data/scene_points.tar.gz"))

with tarfile.open(fileobj=sp_bytes, mode="r:gz") as tar:
    sp_members = tar.getmembers()
    tiff_members = [m for m in sp_members if m.name.endswith(".tiff")]
    print(f"Archivos TIFF dentro del tar: {len(tiff_members)}")
    print(f"Primer frame: {tiff_members[0].name}")
    print(f"Último frame:  {tiff_members[-1].name}")
    print(f"Tamaño típico: {tiff_members[0].size / 1e6:.1f} MB por frame")

    # Leer el primer TIFF de profundidad warpeada
    f_tiff = tar.extractfile(tiff_members[0])
    sp_data = tifffile.imread(io.BytesIO(f_tiff.read()))

print(f"\nForma del TIFF warpeado: {sp_data.shape}")
print(f"Tipo: {sp_data.dtype}")
z_sp = sp_data[..., 2] if sp_data.ndim == 3 else sp_data
valid_sp = (z_sp > 0) & ~np.isnan(z_sp)
print(f"Píxeles válidos: {valid_sp.mean()*100:.1f}%")
print(f"(Los píxeles (0,0,0) son zonas sin GT porque la cámara se movió fuera del área)")

---
## Sección 10 — Vista lado a lado: RGB + Depth

La visualización más útil para el proyecto: la imagen RGB superpuesta con el mapa de profundidad. Las zonas transparentes son exactamente donde el structured light falló (reflejos especulares, bordes).

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

with zipfile.ZipFile(zip_path) as zf:
    img  = read_image(zf, "dataset_1", "keyframe_1", side="Left")
    dept = read_depth(zf, "dataset_1", "keyframe_1", side="left")

# Si las resoluciones difieren, alinear el depth a la imagen
if img.shape[:2] != dept.shape:
    dept = cv2.resize(dept, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Imagen RGB original
axes[0].imshow(img)
axes[0].set_title("RGB original (Left_Image.png)", fontsize=12)
axes[0].axis("off")

# Solo el mapa de depth
vmin, vmax = np.nanpercentile(dept, 2), np.nanpercentile(dept, 98)
im = axes[1].imshow(dept, cmap="plasma", vmin=vmin, vmax=vmax)
axes[1].set_title("Mapa de profundidad GT\n(NaN = sin medición)", fontsize=12)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, label="Profundidad (mm)")

# Overlay: RGB + depth encima (solo donde hay GT válido)
axes[2].imshow(img)
dept_masked = np.ma.masked_where(np.isnan(dept), dept)
axes[2].imshow(dept_masked, cmap="plasma", alpha=0.65, vmin=vmin, vmax=vmax)
axes[2].set_title("Overlay RGB + Depth\n(transparente = sin GT de structured light)", fontsize=12)
axes[2].axis("off")

plt.suptitle("dataset_1 / keyframe_1", fontsize=14)
plt.tight_layout()
plt.show()

---
## Sección 11 — ¿Cuántos keyframes hay en total en todos los datasets?

Un conteo global para saber con cuántos ejemplos de entrenamiento contamos.

In [ ]:
total_kf = 0
print(f"{'Dataset':<12}  {'Keyframes':<40}  Total")
print("-" * 60)
for z in dataset_zips:
    kfs = catalog[z.stem]["keyframes"]
    total_kf += len(kfs)
    print(f"  {z.stem:<10}  {', '.join(kfs):<40}  {len(kfs)}")

print("-" * 60)
print(f"  {'TOTAL':<10}  {'':40}  {total_kf} keyframes")
print()
print("Cada keyframe tiene:")
print("  - 1 imagen RGB (Left_Image.png)         → entrada del modelo")
print("  - 1 mapa de depth GT (left_depth_map)   → supervisión de entrenamiento")
print("  - 1 nube de puntos OBJ                  → GT para Chamfer Distance")
print("  - N cuadros de video con depth warpeado → datos adicionales de entrenamiento")

---
## Resumen final

| Tipo de archivo | Extensión | Qué contiene | Cómo se lee |
|---|---|---|---|
| Imagen RGB | `.png` | Foto del tejido (uint8, 0–255) | `cv2.imdecode` |
| Mapa de profundidad | `.tiff` | Coordenadas XYZ por píxel (float32, mm) | `tifffile.imread` |
| Nube de puntos GT | `.obj` | Vértices 3D en texto | parseo manual de líneas `v ` |
| Calibración | `.yaml` | Matriz intrínseca K (focal, centro óptico) | `cv2.FileStorage` |
| Video de movimiento | `.mp4` | Frames del endoscopio moviendose | `cv2.VideoCapture` |
| Poses por cuadro | `frame_data.tar.gz` | JSONs con transformaciones de cámara | `tarfile` + `json.load` |
| Depths warpeados | `scene_points.tar.gz` | TIFFs de profundidad por cuadro de video | `tarfile` + `tifffile` |

### Lo que aprendimos

1. El dataset tiene **9 ZIPs** con 5 keyframes cada uno — en total **~45 keyframes** de tejido digestivo porcino.
2. Las imágenes tienen resolución **1280×1024 px**.
3. La profundidad GT cubre **~70–87%** de los píxeles; el resto son reflejos especulares o bordes sin medición.
4. El endoscopio opera a **40–130 mm** del tejido dependiendo del keyframe.
5. Todo se puede leer **directamente del ZIP en memoria**, sin extraer nada al disco.

### Siguientes pasos

- `01_eda_iluminacion.ipynb` — Análisis de la iluminación no uniforme (el problema central del proyecto)
- `02_modelos_baseline.ipynb` — Primeros experimentos de corrección de luz